In [13]:
import os 
import math 
import pathlib
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM ,AutoConfig
from datasets import load_dataset

from llmcompressor import oneshot 
from llmcompressor.modifiers.gptq import GPTQModifier

In [5]:
#download the Qwen3-0.6B model from hugging face and put it in folder named model

from pathlib import Path 
from huggingface_hub import snapshot_download 

MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_DIR = Path("models/Qwen3-0.6B")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading model...")


snapshot_download(
    repo_id = MODEL_ID,
    local_dir=str(MODEL_DIR),
)

print("Download Complete!")
print("Saved to:", MODEL_DIR.resolve())

Fetching 10 files: 100%|██████████| 10/10 [01:28<00:00,  8.82s/it]

Download Complete!
Saved to: C:\Users\Anil\Documents\Learning_Projects\local-llm-compression\models\Qwen3-0.6B


In [8]:
from pathlib import Path 
MODEL_DIR = Path("models/Qwen3-0.6B")
print(MODEL_DIR.exists())
print(list(MODEL_DIR.iterdir()))

True
[WindowsPath('models/Qwen3-0.6B/.cache'), WindowsPath('models/Qwen3-0.6B/.gitattributes'), WindowsPath('models/Qwen3-0.6B/config.json'), WindowsPath('models/Qwen3-0.6B/generation_config.json'), WindowsPath('models/Qwen3-0.6B/LICENSE'), WindowsPath('models/Qwen3-0.6B/merges.txt'), WindowsPath('models/Qwen3-0.6B/model.safetensors'), WindowsPath('models/Qwen3-0.6B/README.md'), WindowsPath('models/Qwen3-0.6B/tokenizer.json'), WindowsPath('models/Qwen3-0.6B/tokenizer_config.json'), WindowsPath('models/Qwen3-0.6B/vocab.json')]


In [21]:
config = AutoConfig.from_pretrained(MODEL_DIR)
#print the number of hidden layers
print(config.num_hidden_layers)
print(config.hidden_size)  #tells the dimensionality of the model's hidden representations
print(config.num_attention_heads) #tells the number of attention heads
print(config.head_dim) #tells the head dimension which is how many numerical features each attention head uses for q and k representations
print(config.num_key_value_heads)#how many key-value heads the model has 

#calucalte the GQA 
print(config.num_attention_heads // config.num_key_value_heads)

28
1024
16
128
8
2


In [28]:
text = "I am learning how large language model works"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
tokens = tokenizer.tokenize(text)
print(tokens)

#tokens to id 
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(token_ids)

#convert ids to PyTorch tensor
input_ids = torch.tensor([token_ids])

print(input_ids.shape)


['I', 'Ġam', 'Ġlearning', 'Ġhow', 'Ġlarge', 'Ġlanguage', 'Ġmodel', 'Ġworks']
[40, 1079, 6832, 1246, 3460, 4128, 1614, 4278]
torch.Size([1, 8])


In [ ]:
#load model into CPU 
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.bfloat16, device_map="cpu")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 694.76it/s]


In [ ]:
embedding_layer = model.get_input_embeddings()
print(embedding_layer.weight.shape) #this tells us that the vocab size is 151,936 and numerical value per embedding is 1024

torch.Size([151936, 1024])


In [ ]:
#lets perform the embedding lookup
embeddings = embedding_layer(input_ids)
print(embeddings.shape) #this tells us batch size of 1, sequence length of 8 and embedding dimension of 1024

torch.Size([1, 8, 1024])
